# 02 — Exploracion de Siniestros

## Pregunta de Negocio

¿Cual es el perfil de los siniestros por linea de negocio? ¿Como se distribuyen los montos, los tiempos de reporte y liquidacion?

Exploramos los ~50K siniestros sinteticos calibrados a los agregados del CAS Schedule P.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PALETTE = ['#1E3A5F', '#2B6B4F', '#8B2E3B', '#C4841D', '#5B4A8A', '#4A7FA8']

claims = pd.read_parquet('../data/processed/claims_synthetic.parquet')
claims['accident_year'] = claims['accident_date'].dt.year
print(f'Total siniestros: {len(claims):,}')

## 1. Distribucion de Severidad por LOB

En actuaria, la severidad de siniestros tipicamente sigue una distribucion **lognormal**: la mayoria de los siniestros son pequenos, pero existe una cola pesada de siniestros grandes que domina las perdidas totales.

In [ ]:
fig = px.histogram(
    claims, x='incurred_amount', color='line_of_business',
    nbins=50, log_y=True, barmode='overlay',
    color_discrete_sequence=PALETTE,
    labels={'incurred_amount': 'Monto Incurrido ($)', 'count': 'Frecuencia'},
    title='Distribucion de Severidad por LOB (escala logaritmica)',
    opacity=0.6,
)
fig.update_layout(template='plotly_white', height=450, font_family='Georgia')
fig.show()

In [ ]:
# Statistics by LOB
severity_stats = claims.groupby('line_of_business')['incurred_amount'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).round(0)
severity_stats.columns = ['Count', 'Mean', 'Median', 'Std Dev', 'Min', 'Max']
severity_stats['CV'] = (severity_stats['Std Dev'] / severity_stats['Mean']).round(2)
severity_stats.sort_values('Mean', ascending=False)

**Hallazgo**: El coeficiente de variacion (CV) > 2 en todas las lineas confirma la alta variabilidad tipica de siniestros de seguros. Malpraxis medica y responsabilidad de producto tienen los CVs mas altos — consistente con su naturaleza de baja frecuencia y severidad extrema.

## 2. Bandas de Severidad

Las bandas (Small < $5K, Medium < $25K, Large < $100K, Excess >= $100K) permiten analizar la composicion del portafolio.

In [ ]:
band_order = ['Small', 'Medium', 'Large', 'Excess']
band_summary = claims.groupby(['line_of_business', 'severity_band']).agg(
    count=('claim_id', 'count'),
    total_incurred=('incurred_amount', 'sum'),
).reset_index()

fig = px.bar(
    band_summary, x='line_of_business', y='total_incurred', color='severity_band',
    category_orders={'severity_band': band_order},
    color_discrete_sequence=['#9AB0C8', '#4A7FA8', '#2B527F', '#1E3A5F'],
    labels={'total_incurred': 'Total Incurrido ($)', 'line_of_business': 'Linea de Negocio'},
    title='Composicion de Perdidas por Banda de Severidad',
    barmode='stack',
)
fig.update_layout(template='plotly_white', height=400, font_family='Georgia')
fig.show()

## 3. Retraso de Reporte (Report Lag)

El tiempo entre la fecha del accidente y la fecha de reporte es critico para estimar reservas IBNR. Lineas de cola larga (med mal, product liability) tienen retrasos significativamente mayores.

In [ ]:
fig = px.box(
    claims, x='line_of_business', y='report_lag_days',
    color='line_of_business', color_discrete_sequence=PALETTE,
    labels={'report_lag_days': 'Dias de Retraso', 'line_of_business': ''},
    title='Retraso de Reporte por Linea de Negocio',
)
fig.update_layout(template='plotly_white', height=400, showlegend=False, font_family='Georgia')
fig.show()

In [ ]:
lag_stats = claims.groupby('line_of_business')['report_lag_days'].agg(
    ['mean', 'median', 'std']
).round(0).sort_values('mean', ascending=False)
lag_stats.columns = ['Mean Days', 'Median Days', 'Std Dev']
lag_stats

## 4. Estatus de Siniestros y Tiempo de Liquidacion

In [ ]:
status = claims.groupby(['line_of_business', 'claim_status']).size().unstack(fill_value=0)
status['pct_open'] = (status['Open'] / (status['Open'] + status['Closed']) * 100).round(1)
status.sort_values('pct_open', ascending=False)

In [ ]:
# Distribution by state
state_counts = claims.groupby('state').agg(
    claims=('claim_id', 'count'),
    avg_severity=('incurred_amount', 'mean'),
).sort_values('claims', ascending=False).head(10)

fig = px.bar(
    state_counts.reset_index(), x='state', y='claims',
    color='avg_severity', color_continuous_scale='Blues',
    labels={'claims': 'Numero de Siniestros', 'state': 'Estado', 'avg_severity': 'Severidad Prom.'},
    title='Top 10 Estados por Volumen de Siniestros',
)
fig.update_layout(template='plotly_white', height=350, font_family='Georgia')
fig.show()

## 5. Deteccion de Outliers

Los siniestros con montos superiores al percentil 99 por LOB se consideran outliers potenciales. En actuaria, estos siniestros son criticos — un solo siniestro catastrofico puede distorsionar las reservas.

In [ ]:
outliers = []
for lob, group in claims.groupby('line_of_business'):
    p99 = group['incurred_amount'].quantile(0.99)
    n_outliers = (group['incurred_amount'] > p99).sum()
    total_outlier_amount = group.loc[group['incurred_amount'] > p99, 'incurred_amount'].sum()
    total_amount = group['incurred_amount'].sum()
    outliers.append({
        'LOB': lob,
        'P99 Threshold': f'${p99:,.0f}',
        'N Outliers': n_outliers,
        'Pct of Total Losses': f'{total_outlier_amount/total_amount:.1%}',
    })

pd.DataFrame(outliers)

## So What?

- Las distribuciones lognormales de severidad se confirman: alta asimetria con colas pesadas, especialmente en malpractica medica y responsabilidad de producto.
- El 1% superior de siniestros concentra ~10-25% de las perdidas totales — esto tiene implicaciones directas para reaseguro y capitalizacion.
- Los retrasos de reporte varian drasticamente: auto personal (dias) vs. malpractica medica (meses). Esto explica por que la reserva IBNR es mucho mayor en lineas de cola larga.
- El porcentaje de siniestros abiertos es mas alto en lineas de cola larga, lo que aumenta la incertidumbre en las reservas.